# Local Qwen3 API Notebook

This notebook exposes the cached local model as a small HTTP API and then calls it from inside the notebook.

Model: `Qwen/Qwen3-0.6B`

In [ ]:
%pip install -q transformers torch safetensors fastapi uvicorn requests

In [1]:
from pathlib import Path
import os

import torch
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field
from transformers import AutoModelForCausalLM, AutoTokenizer

# Point directly to the cached Hugging Face snapshot on this machine.
MODEL_DIR = Path.home() / ".cache" / "huggingface" / "hub" / "models--Qwen--Qwen3-0.6B" / "snapshots" / "c1899de289a04d12100db370d81485cdf75e47ca"

# These files are enough to prove the local model is complete and loadable.
required_files = ["config.json", "model.safetensors", "tokenizer.json"]
print(f"MODEL_DIR = {MODEL_DIR}")
print("exists =", MODEL_DIR.exists())
for name in required_files:
    print(f"{name}:", (MODEL_DIR / name).exists())

MODEL_DIR = C:\Users\z5364\.cache\huggingface\hub\models--Qwen--Qwen3-0.6B\snapshots\c1899de289a04d12100db370d81485cdf75e47ca
exists = True
config.json: True
model.safetensors: True
tokenizer.json: True


In [2]:
# Prefer GPU, but fall back to CPU if CUDA is unavailable.
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if DEVICE == "cuda" and torch.cuda.is_bf16_supported() else (torch.float16 if DEVICE == "cuda" else torch.float32)

# Use fewer threads so CPU fallback inference is a bit more predictable on this machine.
torch.set_num_threads(max(1, (os.cpu_count() or 1) // 2))
if DEVICE == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True

# Load tokenizer and model entirely from the local cache. No network access needed.
tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR), trust_remote_code=False)
model = AutoModelForCausalLM.from_pretrained(
    str(MODEL_DIR),
    dtype=DTYPE,
    trust_remote_code=False,
)
model.to(DEVICE)
model.eval()

# Make generation deterministic and keep the demo output clean.
model.generation_config.do_sample = False
model.generation_config.temperature = None
model.generation_config.top_p = None
model.generation_config.top_k = None

print(f"Loaded tokenizer and model on {DEVICE} with dtype={DTYPE}")
print("transformers:", __import__("transformers").__version__)
print("torch:", torch.__version__)

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

Loaded tokenizer and model on CPU
transformers: 4.57.6
torch: 2.7.0+cu128


In [ ]:
# Build a tiny chat wrapper around the local model.
def chat(prompt: str, system: str = "You are a concise and friendly assistant.", max_new_tokens: int = 64) -> str:
    # The model expects messages in chat format, just like a hosted chat API.
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": prompt},
    ]

    # Convert chat messages into token IDs and an attention mask.
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        enable_thinking=False,
        return_dict=True,
        return_tensors="pt",
    )
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

    # Generate the answer on the selected device with the local checkpoint.
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )

    # Strip the prompt tokens so we only keep the assistant reply.
    new_tokens = output_ids[0][inputs["input_ids"].shape[-1] :]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


print(chat("Say hello in one short sentence."))

In [ ]:
# Wrap the local model in a small FastAPI app.
runtime_app = FastAPI(title="Local Qwen3 API", version="1.0")


def ensure_model_files() -> None:
    required = ["config.json", "model.safetensors", "tokenizer.json"]
    missing = [name for name in required if not (MODEL_DIR / name).exists()]
    if missing:
        raise FileNotFoundError(
            f"Missing local model files in {MODEL_DIR}: {', '.join(missing)}"
        )


class ChatRequest(BaseModel):
    prompt: str = Field(..., description="User prompt to send to the local model")
    system: str = Field(
        "You are a concise and friendly assistant.",
        description="System prompt",
    )
    max_new_tokens: int = Field(64, ge=1, le=512)


class ChatResponse(BaseModel):
    model: str
    reply: str
    status: str = "ok"


@runtime_app.get("/health")
def health():
    ensure_model_files()
    return {
        "status": "ok",
        "model_dir": str(MODEL_DIR),
        "model_exists": MODEL_DIR.exists(),
        "device": DEVICE,
        "dtype": str(DTYPE),
    }


@runtime_app.post("/chat", response_model=ChatResponse)
def chat_endpoint(req: ChatRequest) -> ChatResponse:
    try:
        reply = chat(
            prompt=req.prompt,
            system=req.system,
            max_new_tokens=req.max_new_tokens,
        )
    except Exception as exc:
        raise HTTPException(status_code=500, detail=str(exc)) from exc

    return ChatResponse(model="Qwen/Qwen3-0.6B", reply=reply)


print("FastAPI app ready")

In [ ]:
# In-process API test. This does not require starting a separate server.
with TestClient(runtime_app) as client:
    health_result = client.get("/health").json()
    chat_result = client.post(
        "/chat",
        json={
            "prompt": "Say hello in one short sentence.",
            "system": "You are a concise and friendly assistant.",
            "max_new_tokens": 32,
        },
    ).json()

print("HEALTH:", health_result)
print("REPLY:", chat_result["reply"])


In [ ]:
# Client-side helper using a real HTTP request.
def ask_via_api(prompt: str, base_url: str = "http://127.0.0.1:8000", system: str = "You are a concise and friendly assistant.", max_new_tokens: int = 64) -> str:
    import requests

    resp = requests.post(
        f"{base_url}/chat",
        json={
            "prompt": prompt,
            "system": system,
            "max_new_tokens": max_new_tokens,
        },
        timeout=180,
    )
    resp.raise_for_status()
    return resp.json()["reply"]


# Uncomment after starting a real server with uvicorn.
# print(ask_via_api("Say hello in one short sentence."))

If you want to run this as a real HTTP service, start `uvicorn` with `runtime_app`.

The notebook version keeps the same logic but tests it in-process so it is easier to run inside Jupyter.